# B-Tree Visualizer

Interactive step-by-step visualization of B-tree operations including insertions, deletions, splits, and merges.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
from collections import deque
from ipywidgets import interact, interactive, fixed, widgets
from IPython.display import display, clear_output
import copy

In [ ]:
class BTreeNode:
    def __init__(self, is_leaf=False):
        self.keys = []
        self.children = []
        self.is_leaf = is_leaf
        self.parent = None
    
    def __repr__(self):
        return f"Node({self.keys})"

In [ ]:
class BTree:
    def __init__(self, min_degree=3):
        self.min_degree = min_degree
        self.max_keys = 2 * min_degree - 1
        self.min_keys = min_degree - 1
        self.root = BTreeNode(is_leaf=True)
        self.operation_steps = []
        self.current_step = -1
    
    def clone_tree(self):
        """Create a deep copy of the tree for state preservation"""
        cloned = BTree(self.min_degree)
        cloned.root = self._clone_node(self.root)
        return cloned
    
    def _clone_node(self, node):
        """Recursively clone a node and its children"""
        cloned = BTreeNode(node.is_leaf)
        cloned.keys = node.keys.copy()
        cloned.children = [self._clone_node(child) for child in node.children]
        for child in cloned.children:
            child.parent = cloned
        return cloned
    
    def add_step(self, description, highlight_keys=None):
        """Add a step to the operation history"""
        self.operation_steps.append({
            'description': description,
            'tree_state': self.clone_tree(),
            'highlight': highlight_keys if highlight_keys else []
        })
    
    def search(self, node, key):
        """Search for a key in the tree"""
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        if i < len(node.keys) and node.keys[i] == key:
            return node
        if node.is_leaf:
            return None
        return self.search(node.children[i], key)
    
    def insert(self, key):
        """Insert a key into the B-tree with step tracking"""
        self.operation_steps = []
        self.current_step = -1
        
        self.add_step(f"Starting insertion of {key}", [key])
        
        if self.search(self.root, key) is not None:
            self.add_step(f"Key {key} already exists in the tree", [key])
            return
        
        if len(self.root.keys) == self.max_keys:
            self.add_step(f"Root is full ({self.max_keys} keys). Splitting root...")
            new_root = BTreeNode(is_leaf=False)
            new_root.children.append(self.root)
            self.root.parent = new_root
            self.root = new_root
            self.split_child(new_root, 0)
            self.add_step("Root split complete. New root created.")
        
        self.insert_non_full(self.root, key)
        self.add_step(f"Insertion of {key} completed", [key])
        self.current_step = 0
    
    def insert_non_full(self, node, key):
        """Insert key into a non-full node"""
        i = len(node.keys) - 1
        
        if node.is_leaf:
            self.add_step(f"Inserting {key} into leaf node [{', '.join(map(str, node.keys))}]", [key])
            while i >= 0 and node.keys[i] > key:
                i -= 1
            node.keys.insert(i + 1, key)
            self.add_step(f"Key {key} inserted at position {i + 1}", [key])
        else:
            self.add_step(f"Traversing internal node [{', '.join(map(str, node.keys))}]")
            while i >= 0 and node.keys[i] > key:
                i -= 1
            i += 1
            self.add_step(f"Moving to child {i}")
            
            if len(node.children[i].keys) == self.max_keys:
                self.add_step(f"Child node is full. Splitting child at index {i}...")
                self.split_child(node, i)
                self.add_step("Child split complete. Checking if key should go to right child...")
                if node.keys[i] < key:
                    i += 1
            self.insert_non_full(node.children[i], key)
    
    def split_child(self, parent, index):
        """Split a full child node"""
        full_child = parent.children[index]
        new_child = BTreeNode(is_leaf=full_child.is_leaf)
        new_child.parent = parent
        
        self.add_step(f"Splitting node [{', '.join(map(str, full_child.keys))}]", full_child.keys)
        
        mid_index = len(full_child.keys) // 2
        mid_key = full_child.keys[mid_index]
        
        new_child.keys = full_child.keys[mid_index + 1:]
        full_child.keys = full_child.keys[:mid_index]
        
        if not full_child.is_leaf:
            new_child.children = full_child.children[mid_index + 1:]
            full_child.children = full_child.children[:mid_index + 1]
            for child in new_child.children:
                child.parent = new_child
        
        parent.keys.insert(index, mid_key)
        parent.children.insert(index + 1, new_child)
        
        self.add_step(
            f"Split complete: left=[{', '.join(map(str, full_child.keys))}], "
            f"right=[{', '.join(map(str, new_child.keys))}], promoted={mid_key}",
            [mid_key]
        )
    
    def delete(self, key):
        """Delete a key from the B-tree with step tracking"""
        self.operation_steps = []
        self.current_step = -1
        
        self.add_step(f"Starting deletion of {key}", [key])
        
        if self.search(self.root, key) is None:
            self.add_step(f"Key {key} not found in the tree", [key])
            return
        
        self.delete_key(self.root, key)
        
        if len(self.root.keys) == 0 and not self.root.is_leaf:
            self.add_step("Root is empty. Making child the new root...")
            self.root = self.root.children[0]
            self.root.parent = None
            self.add_step("New root set")
        
        self.add_step(f"Deletion of {key} completed", [key])
        self.current_step = 0
    
    def delete_key(self, node, key):
        """Delete key from node"""
        idx = 0
        while idx < len(node.keys) and node.keys[idx] < key:
            idx += 1
        
        if idx < len(node.keys) and node.keys[idx] == key:
            if node.is_leaf:
                self.add_step(f"Found key {key} in leaf node. Removing it...", [key])
                node.keys.pop(idx)
                self.add_step(f"Key {key} removed from leaf", [key])
            else:
                self.add_step(f"Found key {key} in internal node. Replacing with predecessor...", [key])
                self.delete_from_internal_node(node, idx)
        else:
            if node.is_leaf:
                self.add_step(f"Key {key} not found in leaf node")
                return
            
            flag = (idx == len(node.keys))
            self.add_step(f"Key not in current node. Checking child {idx}...")
            
            if len(node.children[idx].keys) < self.min_degree:
                self.add_step(
                    f"Child has insufficient keys ({len(node.children[idx].keys)} < {self.min_degree}). Fixing..."
                )
                self.fill_child(node, idx)
            
            if flag and idx > len(node.keys):
                self.delete_key(node.children[idx - 1], key)
            else:
                self.delete_key(node.children[idx], key)
    
    def delete_from_internal_node(self, node, idx):
        """Delete key from internal node"""
        key = node.keys[idx]
        
        if len(node.children[idx].keys) >= self.min_degree:
            self.add_step("Getting predecessor from left child...")
            pred = self.get_predecessor(node.children[idx])
            self.add_step(f"Predecessor found: {pred}. Replacing key {key}...", [pred])
            node.keys[idx] = pred
            self.delete_key(node.children[idx], pred)
        elif len(node.children[idx + 1].keys) >= self.min_degree:
            self.add_step("Getting successor from right child...")
            succ = self.get_successor(node.children[idx + 1])
            self.add_step(f"Successor found: {succ}. Replacing key {key}...", [succ])
            node.keys[idx] = succ
            self.delete_key(node.children[idx + 1], succ)
        else:
            self.add_step("Both children have minimum keys. Merging children...")
            self.merge_children(node, idx)
            self.delete_key(node.children[idx], key)
    
    def get_predecessor(self, node):
        """Get the predecessor key"""
        while not node.is_leaf:
            node = node.children[-1]
        return node.keys[-1]
    
    def get_successor(self, node):
        """Get the successor key"""
        while not node.is_leaf:
            node = node.children[0]
        return node.keys[0]
    
    def fill_child(self, parent, idx):
        """Ensure child has at least min_degree keys"""
        if idx != 0 and len(parent.children[idx - 1].keys) >= self.min_degree:
            self.add_step("Borrowing from left sibling...")
            self.borrow_from_prev(parent, idx)
        elif idx != len(parent.keys) and len(parent.children[idx + 1].keys) >= self.min_degree:
            self.add_step("Borrowing from right sibling...")
            self.borrow_from_next(parent, idx)
        else:
            self.add_step("Both siblings have minimum keys. Merging with sibling...")
            if idx != len(parent.keys):
                self.merge_children(parent, idx)
            else:
                self.merge_children(parent, idx - 1)
    
    def borrow_from_prev(self, parent, idx):
        """Borrow a key from previous sibling"""
        child = parent.children[idx]
        sibling = parent.children[idx - 1]
        
        child.keys.insert(0, parent.keys[idx - 1])
        if not child.is_leaf:
            child.children.insert(0, sibling.children.pop())
            child.children[0].parent = child
        parent.keys[idx - 1] = sibling.keys.pop()
        
        self.add_step(f"Borrowed key from left sibling. Parent key {parent.keys[idx - 1]} moved down.", 
                     [parent.keys[idx - 1]])
    
    def borrow_from_next(self, parent, idx):
        """Borrow a key from next sibling"""
        child = parent.children[idx]
        sibling = parent.children[idx + 1]
        
        child.keys.append(parent.keys[idx])
        if not child.is_leaf:
            child.children.append(sibling.children.pop(0))
            child.children[-1].parent = child
        parent.keys[idx] = sibling.keys.pop(0)
        
        self.add_step(f"Borrowed key from right sibling. Parent key {parent.keys[idx]} moved down.", 
                     [parent.keys[idx]])
    
    def merge_children(self, parent, idx):
        """Merge child with its next sibling"""
        child = parent.children[idx]
        sibling = parent.children[idx + 1]
        
        self.add_step(
            f"Merging nodes: [{', '.join(map(str, child.keys))}] + {parent.keys[idx]} + "
            f"[{', '.join(map(str, sibling.keys))}]",
            [parent.keys[idx]]
        )
        
        child.keys.append(parent.keys[idx])
        child.keys.extend(sibling.keys)
        
        if not child.is_leaf:
            for c in sibling.children:
                c.parent = child
                child.children.append(c)
        
        parent.keys.pop(idx)
        parent.children.pop(idx + 1)
        
        self.add_step(f"Merge complete: [{', '.join(map(str, child.keys))}]")

In [ ]:
def calculate_layout(tree):
    """Calculate a better layout for the tree"""
    positions = {}
    
    def get_subtree_width(node, level=0):
        if node.is_leaf:
            return len(node.keys) * 0.6 + 0.3
        else:
            total = sum(get_subtree_width(child, level + 1) for child in node.children)
            return max(total, len(node.keys) * 0.6 + 0.3)
    
    def assign_positions(node, x_offset, level=0):
        if node.is_leaf:
            node_width = len(node.keys) * 0.6 + 0.3
            positions[id(node)] = (x_offset + node_width / 2, -level * 2.5)
            return node_width
        else:
            child_widths = []
            current_x = x_offset
            
            for child in node.children:
                width = assign_positions(child, current_x, level + 1)
                child_widths.append(width)
                current_x += width + 0.2
            
            total_width = current_x - x_offset - 0.2
            node_x = x_offset + total_width / 2
            positions[id(node)] = (node_x, -level * 2.5)
            return total_width
    
    assign_positions(tree.root, 0)
    return positions

In [ ]:
def visualize_tree(tree, highlight_keys=None, step_description="", figsize=(14, 8)):
    """Visualize the B-tree with highlighted keys"""
    if highlight_keys is None:
        highlight_keys = []
    
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect('equal')
    ax.axis('off')
    
    positions = calculate_layout(tree)
    
    # Draw edges first
    def draw_edges(node):
        if not node.is_leaf:
            node_pos = positions[id(node)]
            for child in node.children:
                child_pos = positions[id(child)]
                ax.plot([node_pos[0], child_pos[0]], [node_pos[1], child_pos[1]], 
                       'k-', linewidth=1.5, alpha=0.6, zorder=1)
                draw_edges(child)
    
    draw_edges(tree.root)
    
    # Draw nodes
    for node_id, (x, y) in positions.items():
        # Find the node
        node = None
        def find_node(n):
            nonlocal node
            if id(n) == node_id:
                node = n
                return
            for child in n.children:
                find_node(child)
        find_node(tree.root)
        
        if node is None:
            continue
        
        # Calculate node width based on number of keys
        num_keys = len(node.keys)
        node_width = num_keys * 0.6 + 0.3
        node_height = 0.8
        
        # Draw node box
        box = FancyBboxPatch(
            (x - node_width/2, y - node_height/2), node_width, node_height,
            boxstyle="round,pad=0.1",
            edgecolor='#2c3e50',
            facecolor='#ecf0f1' if not node.is_leaf else '#e8f5e9',
            linewidth=2,
            zorder=2
        )
        ax.add_patch(box)
        
        # Draw keys
        key_width = 0.5
        start_x = x - (num_keys - 1) * key_width / 2
        
        for i, key in enumerate(node.keys):
            key_x = start_x + i * key_width
            
            # Highlight if in highlight list
            if key in highlight_keys:
                highlight_box = FancyBboxPatch(
                    (key_x - 0.25, y - 0.3), 0.5, 0.6,
                    boxstyle="round,pad=0.05",
                    edgecolor='#e74c3c',
                    facecolor='#ffebee',
                    linewidth=2.5,
                    zorder=3
                )
                ax.add_patch(highlight_box)
            
            # Draw key text
            ax.text(key_x, y, str(key), 
                   ha='center', va='center',
                   fontsize=11, fontweight='bold',
                   color='#2c3e50', zorder=4)
    
    # Add title with step description
    if step_description:
        ax.text(0.5, 0.98, step_description, 
               transform=ax.transAxes,
               ha='center', va='top',
               fontsize=12, fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Initialize the B-tree
btree = BTree(min_degree=3)

# Global state for step navigation
current_operation_steps = []
current_step_index = -1

In [ ]:
def show_current_step():
    """Display the current step of the operation"""
    global current_operation_steps, current_step_index, btree
    
    if not current_operation_steps or current_step_index < 0:
        # Show current tree state
        visualize_tree(btree, step_description="Current B-Tree State")
        return
    
    if current_step_index >= len(current_operation_steps):
        current_step_index = len(current_operation_steps) - 1
    
    step = current_operation_steps[current_step_index]
    tree_state = step['tree_state']
    description = step['description']
    highlight = step.get('highlight', [])
    
    step_info = f"Step {current_step_index + 1} of {len(current_operation_steps)}"
    visualize_tree(tree_state, highlight_keys=highlight, 
                  step_description=f"{step_info}: {description}")

In [ ]:
def insert_key(key):
    """Insert a key and prepare for step-by-step visualization"""
    global current_operation_steps, current_step_index, btree
    
    btree.insert(key)
    current_operation_steps = btree.operation_steps
    current_step_index = 0
    
    show_current_step()

def delete_key(key):
    """Delete a key and prepare for step-by-step visualization"""
    global current_operation_steps, current_step_index, btree
    
    btree.delete(key)
    current_operation_steps = btree.operation_steps
    current_step_index = 0
    
    show_current_step()

def next_step():
    """Move to next step"""
    global current_step_index
    if current_operation_steps and current_step_index < len(current_operation_steps) - 1:
        current_step_index += 1
        show_current_step()

def prev_step():
    """Move to previous step"""
    global current_step_index
    if current_step_index > 0:
        current_step_index -= 1
        show_current_step()

def reset_tree(min_degree=3):
    """Reset the B-tree"""
    global btree, current_operation_steps, current_step_index
    btree = BTree(min_degree=min_degree)
    current_operation_steps = []
    current_step_index = -1
    show_current_step()

In [ ]:
# Create interactive widgets
def create_controls():
    """Create interactive control panel"""
    
    min_degree_slider = widgets.IntSlider(
        value=3,
        min=2,
        max=5,
        step=1,
        description='Min Degree:',
        style={'description_width': 'initial'}
    )
    
    value_input = widgets.IntText(
        value=10,
        description='Value:',
        style={'description_width': 'initial'}
    )
    
    insert_btn = widgets.Button(
        description='Insert',
        button_style='success',
        icon='plus'
    )
    
    delete_btn = widgets.Button(
        description='Delete',
        button_style='danger',
        icon='minus'
    )
    
    prev_btn = widgets.Button(
        description='← Previous Step',
        button_style='info'
    )
    
    next_btn = widgets.Button(
        description='Next Step →',
        button_style='info'
    )
    
    reset_btn = widgets.Button(
        description='Reset Tree',
        button_style='warning'
    )
    
    output = widgets.Output()
    
    def on_insert_click(b):
        with output:
            clear_output(wait=True)
            insert_key(value_input.value)
    
    def on_delete_click(b):
        with output:
            clear_output(wait=True)
            delete_key(value_input.value)
    
    def on_prev_click(b):
        with output:
            clear_output(wait=True)
            prev_step()
    
    def on_next_click(b):
        with output:
            clear_output(wait=True)
            next_step()
    
    def on_reset_click(b):
        with output:
            clear_output(wait=True)
            reset_tree(min_degree_slider.value)
    
    insert_btn.on_click(on_insert_click)
    delete_btn.on_click(on_delete_click)
    prev_btn.on_click(on_prev_click)
    next_btn.on_click(on_next_click)
    reset_btn.on_click(on_reset_click)
    
    controls = widgets.VBox([
        widgets.HBox([min_degree_slider, reset_btn]),
        widgets.HBox([value_input, insert_btn, delete_btn]),
        widgets.HBox([prev_btn, next_btn]),
        output
    ])
    
    return controls

# Display controls and initial tree
controls = create_controls()
display(controls)
show_current_step()

## Usage Instructions

1. **Set Minimum Degree**: Use the slider to set the minimum degree (t) of the B-tree. This determines the maximum number of keys per node (2t-1).

2. **Insert a Key**: Enter a value in the "Value" field and click "Insert" to insert it into the tree. The visualization will show all steps including splits.

3. **Delete a Key**: Enter a value and click "Delete" to remove it from the tree. The visualization will show all steps including merges and borrows.

4. **Navigate Steps**: Use "Previous Step" and "Next Step" buttons to move through the operation history step-by-step.

5. **Reset Tree**: Click "Reset Tree" to start with a fresh empty tree.

### Visual Features:
- **Green nodes**: Leaf nodes
- **Gray nodes**: Internal nodes
- **Red highlight**: Keys being operated on in the current step
- **Step description**: Shows what operation is happening at each step

In [ ]:
# Example: Insert a sequence of keys to see splits
print("Example: Inserting keys 10, 20, 30, 40, 50, 60, 70, 80, 90...")
for key in [10, 20, 30, 40, 50, 60, 70, 80, 90]:
    btree.insert(key)

current_operation_steps = btree.operation_steps
current_step_index = len(current_operation_steps) - 1
show_current_step()